# Tensors and autograd

A tensor is PyTorch's n-dimensional array. **Autograd** records the operations applied to tensors so it can compute gradients for you. This notebook covers both, ending with gradient descent written by hand.

In [ ]:
import torch

print("PyTorch", torch.__version__)

## Creating tensors

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
zeros = torch.zeros(2, 3)
torch.manual_seed(0)
noise = torch.randn(2, 3)

print("a.shape:", a.shape, " dtype:", a.dtype)
print("zeros:\n", zeros)
print("noise:\n", noise)

## Operations and broadcasting

Arithmetic is element-wise. When shapes differ, PyTorch **broadcasts** the smaller tensor across the larger one. `@` is matrix multiplication.

In [ ]:
print("a + 10:\n", a + 10)
print("a * [10, 100] (broadcast over rows):\n", a * torch.tensor([10.0, 100.0]))
print("a @ a:\n", a @ a)

assert (a @ a).tolist() == [[7.0, 10.0], [15.0, 22.0]]

## Autograd

Mark a tensor with `requires_grad=True`, compute something from it, and call `.backward()`. The gradient lands in `.grad`.

For $y = x^2 + 3x$ the derivative is $2x + 3$, which is 7 at $x = 2$.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x
y.backward()

print("dy/dx at x=2:", x.grad.item())
assert x.grad.item() == 7.0

## Gradient descent by hand

Minimise $f(p) = (p - 5)^2$. Each step moves $p$ against its gradient. Updates happen under `torch.no_grad()` so they are not recorded as part of the graph, and the gradient is cleared each step because `.backward()` accumulates.

In [ ]:
p = torch.tensor(0.0, requires_grad=True)
lr = 0.1
history = []

for step in range(60):
    loss = (p - 5.0) ** 2
    loss.backward()
    with torch.no_grad():
        p -= lr * p.grad
    p.grad.zero_()
    history.append(p.item())

print(f"p after 60 steps: {p.item():.6f}  (true minimum: 5)")
assert abs(p.item() - 5.0) < 1e-3

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.plot(history)
plt.axhline(5.0, color="gray", linestyle="--", label="minimum")
plt.xlabel("step")
plt.ylabel("p")
plt.title("Gradient descent on (p - 5)^2")
plt.legend()
plt.tight_layout()
plt.show()

## Next

`02_linear_regression.ipynb` replaces the hand-written update with an `nn.Module` and an optimizer.